# Queimar Legenda MULTICOR — 5 Idiomas (classificação gramatical por palavra)

Pega um arquivo `.ass` (o que saiu do `gerar-legenda-colorida.ipynb` — pode
ser o original ou uma versão que você corrigiu manualmente) e queima no
vídeo/imagem de fundo do vídeo.

No final, duas ações **separadas**: baixar o vídeo final pra conferir, e
(só depois de confirmar que ficou bom) salvar no Drive.


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╗
# Garante a fonte do coreano (Noto Sans CJK) instalada — sem isso, o
# libass (o que desenha a legenda em cima do vídeo) não acha os
# caracteres Hangul e mostra quadradinho (□) no lugar. Não dá pra supor
# que já vem instalada no ambiente do Colab.
!apt-get install -y -qq fonts-noto-cjk > /dev/null 2>&1

import shutil, sys
from pathlib import Path
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

print("✅ Setup concluído")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

# Nome do vídeo/imagem de fundo (o mesmo que o resto do pipeline gera) —
# ajuste se o nome for outro
NOME_VIDEO_BASE = f"{NOME_ORACAO}_video_base.mp4"

from config import PipelineConfig
from drive_utils import DriveClient

config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE="en")
drive_client = DriveClient.get()

print(f"Vídeo: {NOME_ORACAO}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. ENVIAR O .ASS (o original do notebook anterior, ou uma       ║
# ║  versão que você corrigiu manualmente — tanto faz, contanto que  ║
# ║  seja um .ass válido)                                            ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files

print("Selecione o arquivo .ass:")
enviados = files.upload()
nome_ass = list(enviados.keys())[0]
caminho_ass = Path(nome_ass)
print(f"✅ Recebido: {caminho_ass}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. BAIXAR O VÍDEO/IMAGEM DE FUNDO DO DRIVE                      ║
# ╚══════════════════════════════════════════════════════════════════╝
from srt_utils import ler_srt  # garante que os módulos do pipeline já foram testados no import

video_base_local = Path(NOME_VIDEO_BASE)
ok = drive_client.download(config.pasta_oracao, NOME_VIDEO_BASE, video_base_local)
if not ok:
    raise FileNotFoundError(f"Não achei '{NOME_VIDEO_BASE}' em {config.pasta_oracao} — ajuste NOME_VIDEO_BASE na célula 2")
print(f"✅ Vídeo base baixado: {video_base_local}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. QUEIMAR                                                      ║
# ╚══════════════════════════════════════════════════════════════════╗
from ffmpeg_utils import queimar_legendas_ass

caminho_saida = Path(f"{NOME_ORACAO}_com_legenda_colorida.mp4")
resultado = queimar_legendas_ass(
    video_entrada=video_base_local,
    ass_path=caminho_ass,
    saida=caminho_saida,
)
print(f"✅ Vídeo com legenda colorida: {resultado}")
print("\nRode a célula 6 pra baixar e conferir. Só rode a célula 7 (salvar no Drive) depois de confirmar que ficou bom.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. BAIXAR O RESULTADO (pra conferir)                            ║
# ╚══════════════════════════════════════════════════════════════════╝
files.download(str(resultado))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(resultado, config.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
